# Lesson 04 - Patrón de Diseño de Uso de Herramientas

En esta lección aprenderás el patrón de diseño **Uso de Herramientas** para agentes de IA usando el Microsoft Agent Framework (Python). Cubrimos:

- Definir herramientas de función con el decorador `@tool` y parámetros tipados
- Proporcionar esquemas de herramientas para que el modelo sepa qué hace cada herramienta
- Controlar la ejecución de herramientas con `approval_mode`
- Devolver **salida estructurada** mediante modelos Pydantic y `response_format`

El escenario es un **agente de reserva de viajes** que puede buscar destinos, verificar disponibilidad y recuperar información de vuelos.


## Configuración


In [ ]:
! pip install agent-framework azure-ai-projects azure-identity -U -q

In [ ]:
import logging
logging.getLogger("agent_framework.azure").setLevel(logging.ERROR)

import os
import asyncio
from typing import Annotated

from pydantic import BaseModel
from agent_framework import tool
from agent_framework.azure import AzureAIProjectAgentProvider
from azure.identity import AzureCliCredential

In [ ]:
# Create the Azure AI Foundry provider
provider = AzureAIProjectAgentProvider(credential=AzureCliCredential())

## Definiendo Herramientas con el Decorador @tool

El decorador `@tool` convierte una función Python simple en una herramienta que un agente puede llamar.  
Puntos clave:

- El **docstring** se convierte en la descripción de la herramienta que el modelo ve.
- Las **anotaciones de tipo** (incluyendo `Annotated` con descripciones) definen el esquema de la herramienta.
- `approval_mode` controla si el usuario debe aprobar cada llamada antes de que se ejecute.


In [ ]:
@tool(approval_mode="never_require")
def get_destinations() -> list[str]:
    """Get available vacation destinations."""
    return ["Barcelona", "Paris", "Berlin", "Tokyo", "Sydney", "New York City"]


@tool(approval_mode="never_require")
def check_availability(
    destination: Annotated[str, "The destination to check"],
) -> str:
    """Check booking availability for a destination."""
    availability = {
        "Barcelona": "Available - 3 spots left",
        "Paris": "Available",
        "Berlin": "Sold out",
        "Tokyo": "Available - 1 spot left",
        "Sydney": "Available",
        "New York City": "Available",
    }
    return availability.get(destination, "Unknown destination")


@tool(approval_mode="never_require")
def get_flight_info(
    origin: Annotated[str, "Origin airport code"],
    destination: Annotated[str, "Destination airport code"],
) -> str:
    """Get flight information between two cities."""
    flights = {
        "LHR-BCN": "BA 2042, Departs 08:30, Arrives 11:45, $350",
        "LHR-CDG": "AF 1081, Departs 09:15, Arrives 11:30, $280",
        "LHR-NRT": "JL 044, Departs 11:00, Arrives 07:00+1, $890",
    }
    return flights.get(
        f"{origin}-{destination}",
        f"No direct flights from {origin} to {destination}",
    )

## Crear un Agente con Múltiples Herramientas

Pasa las tres herramientas al cliente para que el modelo pueda invocar las que necesite para responder a la pregunta del usuario.


In [ ]:
travel_tools = [get_destinations, check_availability, get_flight_info]

agent = await provider.create_agent(
    name="TravelToolAgent",
    instructions="You are a travel agent. Use the available tools to answer questions about destinations, availability, and flights.",
    tools=travel_tools,
)

response = await agent.run(
    "What destinations do you have? Which ones are still available?"
)
print(response)

## Salida Estructurada con Herramientas

Al establecer `response_format` en un modelo Pydantic, se obliga al agente a devolver un objeto JSON bien tipado en lugar de texto libre. Esto es útil cuando el código posterior necesita consumir el resultado de forma programática.


In [ ]:
class BookingRecommendation(BaseModel):
    destination: str
    available: bool
    flight_details: str
    estimated_cost: int


class TravelPlan(BaseModel):
    recommendations: list[BookingRecommendation]


structured_agent = await provider.create_agent(
    name="StructuredTravelAgent",
    instructions=(
        "You are a travel agent. Use the available tools to find destinations, "
        "check availability, and get flight info. Return structured results."
    ),
    tools=[get_destinations, check_availability, get_flight_info],
)

response = await structured_agent.run(
    "I want to fly from London Heathrow to somewhere warm in Europe. "
    "Check what's available."
)
if response:
    print(response)

## Patrones de Aprobación de Herramientas

El parámetro `approval_mode` en `@tool` controla si las llamadas a herramientas requieren aprobación humana antes de ejecutarse:

| Modo | Comportamiento |
|---|---|
| `"never_require"` | La herramienta se ejecuta automáticamente — no se necesita confirmación del usuario. |
| `"always_require"` | Cada llamada debe ser aprobada por el usuario antes de ejecutarse. |

Usa `"always_require"` para herramientas que tienen efectos secundarios (por ejemplo, reservar un vuelo, cobrar una tarjeta de crédito) para que un humano esté involucrado en el proceso.


In [ ]:
@tool(approval_mode="always_require")
def book_flight(
    origin: Annotated[str, "Origin airport code"],
    destination: Annotated[str, "Destination airport code"],
    passenger_name: Annotated[str, "Full name of the passenger"],
) -> str:
    """Book a flight for a passenger. Requires approval before executing."""
    return (
        f"Flight booked from {origin} to {destination} "
        f"for {passenger_name}. Confirmation #TRV-2024-{hash(passenger_name) % 10000:04d}"
    )


print("Tool name:", book_flight.name)
print("Approval mode:", book_flight.approval_mode)

## Summary

En esta lección aprendiste a:

1. **Definir herramientas** usando el decorador `@tool` con parámetros tipados y docstrings que sirven como el esquema de la herramienta.  
2. **Componer múltiples herramientas** para que el agente pueda llamarlas en secuencia y responder consultas complejas.  
3. **Devolver una salida estructurada** pasando un modelo Pydantic como `response_format`.  
4. **Controlar la aprobación de herramientas** con `approval_mode` para mantener a un humano en el proceso en operaciones sensibles.  

Estos patrones forman la base para construir agentes confiables, listos para producción, que puedan interactuar de forma segura con sistemas externos.


---

<!-- CO-OP TRANSLATOR DISCLAIMER START -->
**Aviso legal**:  
Este documento ha sido traducido utilizando el servicio de traducción automática [Co-op Translator](https://github.com/Azure/co-op-translator). Aunque nos esforzamos por la precisión, tenga en cuenta que las traducciones automáticas pueden contener errores o inexactitudes. El documento original en su idioma nativo debe considerarse la fuente autorizada. Para información crítica, se recomienda la traducción profesional realizada por humanos. No nos hacemos responsables de cualquier malentendido o interpretación errónea que pueda surgir del uso de esta traducción.
<!-- CO-OP TRANSLATOR DISCLAIMER END -->
